# Project 16 — Quantization Lab (BF16 / FP8 / INT8 / INT4 / NVFP4-sim)

```
TinyShakespeare-trained TinyGPT (L2/d64, brief, CPU)
  -> same weights, 7 numeric formats: FP32 | BF16 | FP16 | FP8-E4M3(fake) | INT8 sym | INT4 group-g32 | NVFP4-sim (INT4 g16)
  -> one table: memory (MB) | val CE / ppl | fwd latency | tok/s
  -> what-if: INT4 group-size sweep (per-tensor -> g128 -> g32 -> g16) + clip ablation
```

Goal: feel the memory/quality knob of every format with real numbers, and learn
why block granularity (not just bit-width) decides INT4 quality.

Learning objectives:
1. Write the affine fake-quant math (`q = clamp(round(w/s)+z)`, `w' = s*(q-z)`) from memory.
2. State bytes/param per format and the KV-cache side of the same math.
3. Explain why BF16 is lossless-ish, FP8 needs scaling, and INT4 needs groups (outliers).
4. Read a memory-vs-quality frontier and pick a format for a memory budget.
   Key lesson: BF16 halves memory for ~free; INT8 quarters it for ~free; INT4 needs
   small groups (g32/g16) or quality falls off a cliff.


In [ ]:
import os # need os to make results/ and read cpu count
import random # need random to seed stdlib RNG
import time # need time to measure forward latency per format
import csv # need csv to write quant.csv + groups csv
import math # need math for perplexity exp() and scale math
import urllib.request # need urllib to try downloading TinyShakespeare (offline fallback)
import numpy as np # need numpy for batch indexing and error stats
import torch # need torch for TinyGPT + fake-quant casts
import torch.nn.functional as F # need F for cross-entropy val loss
import matplotlib # need matplotlib for frontier plots
matplotlib.use('Agg') # force headless PNG backend (no display needed)
import matplotlib.pyplot as plt # need pyplot for memory-vs-loss figure
SEED = 16 # fixed seed: same init, same data order, fair formats
random.seed(SEED) # seed stdlib
np.random.seed(SEED) # seed numpy
torch.manual_seed(SEED) # seed torch
torch.set_num_threads(max(1, os.cpu_count() // 2)) # cap threads: stable timing on shared boxes
os.makedirs('./results', exist_ok=True) # create results/ for CSVs and PNGs
DEV = 'cuda' if torch.cuda.is_available() else 'cpu' # GPU if present else CPU (bench notes device)
print(f"device={DEV} torch={torch.__version__}") # env line for the README


In [ ]:
# ---- data: TinyShakespeare download with offline synthetic fallback ----
URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt' # canonical tiny-shakespeare mirror (same as P11/P13)
try: # try the download first (cached corpora make reruns offline-capable)
    raw, _ = urllib.request.urlretrieve(URL, './results/tinyshakespeare.txt') # download to results/ (doubles as cache)
    text = open(raw, encoding='utf-8').read() # read the downloaded text
    print(f"downloaded chars={len(text)}") # confirm live corpus
except Exception as e: # offline sandbox: fall back to deterministic synthetic text
    print(f"download failed ({e}); using synthetic fallback") # say why we fell back
    rng = np.random.RandomState(SEED) # seeded RNG: same fallback every run
    words = ['king', 'queen', 'thou', 'thee', 'night', 'light', 'love', 'hate', 'fire', 'water'] # tiny vocab with repeat structure (learnable)
    text = ' '.join(rng.choice(words, size=60000).tolist()) # 60k pseudo-words: same pipeline, no network
    print(f"synthetic chars={len(text)}") # confirm fallback size
chars = sorted(set(text)) # vocab = sorted unique chars (char-level tokenizer)
V = len(chars) # vocab size drives ln(V) = init-loss baseline
stoi = {c: i for i, c in enumerate(chars)} # char -> id map
itos = {i: c for c, i in stoi.items()} # id -> char map
data = torch.tensor([stoi[c] for c in text], dtype=torch.long) # full corpus as ids
n = int(0.9 * len(data)) # 90/10 split point
tr, va = data[:n], data[n:] # train / val id tensors
CTX = 64 # context length: short keeps CPU bench fast
def batch(split, b): # random-window batch sampler for train/val
    src = tr if split == 'tr' else va # pick the split tensor
    ix = torch.randint(0, len(src) - CTX - 1, (b,)) # random offsets with replacement
    x = torch.stack([src[i:i + CTX] for i in ix]) # inputs: (b, CTX) windows
    y = torch.stack([src[i + 1:i + CTX + 1] for i in ix]) # labels: same windows shifted by one
    return x, y # (inputs, next-token labels)
print(f"vocab={V} lnV={math.log(V):.4f} train={len(tr)} val={len(va)}") # dataset card (init CE should sit near lnV)


In [ ]:
# ---- model: TinyGPT (L2/d64, same recipe as P13-S) + fake-quant library ----
class Block(torch.nn.Module): # one pre-norm transformer block (attn + mlp)
    def __init__(self, d, h): # d = width, h = heads
        super().__init__() # init nn.Module bookkeeping
        self.ln1 = torch.nn.LayerNorm(d) # pre-attn norm
        self.attn = torch.nn.MultiheadAttention(d, h, batch_first=True) # stock MHA (block factory, not the lesson)
        self.ln2 = torch.nn.LayerNorm(d) # pre-mlp norm
        self.fc1 = torch.nn.Linear(d, 4 * d) # mlp expand x4
        self.fc2 = torch.nn.Linear(4 * d, d) # mlp project back
    def forward(self, x): # x: (b, t, d)
        a, _ = self.attn(self.ln1(x), self.ln1(x), self.ln1(x), need_weights=False) # self-attn on normed x (no weights out)
        x = x + a # residual 1
        return x + self.fc2(torch.nn.functional.gelu(self.fc1(self.ln2(x)))) # residual 2 around mlp
class TinyGPT(torch.nn.Module): # char-level GPT: embed -> blocks -> head
    def __init__(self, L, d, h, v, ctx): # L layers, d width, h heads, v vocab, ctx length
        super().__init__() # init bookkeeping
        self.tok = torch.nn.Embedding(v, d) # token embedding table
        self.pos = torch.nn.Embedding(ctx, d) # learned absolute positions
        self.blocks = torch.nn.ModuleList([Block(d, h) for _ in range(L)]) # L stacked blocks
        self.ln = torch.nn.LayerNorm(d) # final norm
        self.head = torch.nn.Linear(d, v, bias=False) # LM head (no bias: fewer params)
    def forward(self, x): # x: (b, t) ids -> (b, t, v) logits
        b, t = x.shape # read batch, time
        h = self.tok(x) + self.pos(torch.arange(t, device=x.device)) # tok + pos embeddings broadcast over batch
        for blk in self.blocks: # run the stack
            h = blk(h) # one block
        return self.head(self.ln(h)) # norm then project to vocab
def count_params(m): # exact param count (asserted against analytic formula in P13 style)
    return sum(p.numel() for p in m.parameters()) # sum over every tensor
# -- fake-quant core: affine map q = clamp(round(w/s) + z), dequant w' = s*(q - z) --
def _per_tensor_scale(w, bits): # symmetric per-tensor scale from max abs (1 sign bit assumed)
    mx = w.abs().max().clamp_min(1e-8) # peak magnitude; floor avoids div-by-zero on zero tensors
    return mx / float(2 ** (bits - 1) - 1) # scale so max maps to top code
def fake_int8(w, clip=None): # symmetric INT8 fake-quant, optional percentile clip (outlier control)
    wc = w # default: no clipping
    if clip is not None: # clip mode: bound range before choosing scale
        q = torch.quantile(w.abs().flatten().float(), clip) # e.g. 0.99 -> 99th pct of |w|
        qq = q.clamp_min(1e-8) # floored clip value reused below
        wc = w.clamp(-qq, qq) # clamp symmetric at that quantile
    s = _per_tensor_scale(wc, 8) # one scale for the whole tensor
    q = (wc / s).round().clamp(-128, 127) # quantize to int8 codes
    return (q * s).to(w.dtype) # dequantize back (fake: compute still in fp32)
def fake_int4_group(w, group=32, clip=None): # INT4 with per-group scales along last dim (granularity knob)
    orig = w.shape # remember shape to restore
    flat1 = w.reshape(-1) # flatten: padding below handles sizes not divisible by group
    pad = (-flat1.numel()) % group # missing elements to reach a multiple of group
    flat = torch.cat([flat1, flat1[:1].repeat(pad)]) if pad else flat1 # pad by repeating the first element (trimmed after dequant)
    flat = flat.reshape(-1, group) # carve into groups of `group` elements
    if clip is not None: # optional per-group percentile clip
        q = torch.quantile(flat.abs().float(), clip, dim=1, keepdim=True) # (G,1) quantiles
        flat = flat.clamp(-q.clamp_min(1e-8), q.clamp_min(1e-8)) # symmetric clamp per group
    mx = flat.abs().max(dim=1, keepdim=True).values.clamp_min(1e-8) # per-group peak
    s = mx / 7.0 # 4-bit signed max code is 7
    q = (flat / s).round().clamp(-8, 7) # 4-bit codes
    dq = (q * s).reshape(-1)[:flat1.numel()] # dequantize, drop padding
    return dq.reshape(orig).to(w.dtype) # restore shape
def fake_float_cast(w, dtype): # BF16/FP16/FP8 fake-quant: cast down then back (range/precision loss only)
    return w.to(dtype).to(torch.float32) # round-trip through the narrow float
def quantize_state(model, fn): # apply a weight transform to a cloned model, return it
    import copy # need copy to clone the model without touching the baseline
    m2 = copy.deepcopy(model) # deep copy: baseline weights stay fp32
    with torch.no_grad(): # no graph for the weight rewrite
        for name, p in m2.named_parameters(): # visit every weight tensor
            p.copy_(fn(p.data.float())) # replace with fake-quantized values
    return m2 # quantized clone ready for eval
# ---- unit tests: error ordering + memory ordering (fail fast if quant math is wrong) ----
_t = torch.randn(512, 512) * 2.0 # probe tensor with realistic spread
assert (fake_int8(_t) - _t).abs().mean() < (fake_int4_group(_t, 32) - _t).abs().mean() # int8 must beat int4-g32 on mean abs error
assert (fake_int4_group(_t, 16) - _t).abs().mean() <= (fake_int4_group(_t, 128) - _t).abs().mean() # finer groups must not lose to coarser ones
assert (fake_float_cast(_t, torch.bfloat16) - _t).abs().mean() < (fake_int8(_t) - _t).abs().mean() # bf16 ~lossless: must beat int8 on fp32-range probe
print("quant unit tests passed (int8<int4 err; g16<=g128 err; bf16<int8 err)") # confirm library sane


In [ ]:
# ---- baseline: brief pretrain, then one table across 7 formats ----
model = TinyGPT(2, 64, 4, V, CTX).to(DEV) # tiny proxy: L2/d64 (~110k params, seconds on CPU)
P = count_params(model) # exact param count for the memory column
opt = torch.optim.AdamW(model.parameters(), lr=3e-4) # standard LM optimizer
for it in range(60): # brief pretrain: learn letters/rhythm, not mastery (keeps CPU runtime ~seconds)
    xb, yb = batch('tr', 32) # random train windows
    xb, yb = xb.to(DEV), yb.to(DEV) # to device
    loss = F.cross_entropy(model(xb).view(-1, V), yb.view(-1)) # next-token CE
    opt.zero_grad() # clear grads
    loss.backward() # backprop
    opt.step() # update
print(f"baseline trained: params={P:,} last_loss={loss.item():.4f} (init~{math.log(V):.2f})") # training card
def val_ce(m, n=20): # mean val CE over n batches (no grad)
    m.eval() # eval mode (no dropout anyway)
    tot, cnt = 0.0, 0 # accumulate loss mass + token count
    with torch.no_grad(): # no graph
        for _ in range(n): # n val batches
            xb, yb = batch('va', 32) # val windows
            xb, yb = xb.to(DEV), yb.to(DEV) # to device
            l = F.cross_entropy(m(xb).view(-1, V), yb.view(-1)) # batch CE
            tot += l.item() * xb.numel() # weight by tokens
            cnt += xb.numel() # count tokens
    m.train() # restore train mode
    return tot / cnt # mean CE per token
def bench_fwd(m, iters=30, b=8): # median forward latency (sync-safe) + tok/s
    xb, _ = batch('va', b) # fixed-ish val batch shape (b, CTX)
    xb = xb.to(DEV) # to device once
    for _ in range(5): # warmup: settle caches/clocks
        _ = m(xb) # throwaway forward
    if DEV == 'cuda': # GPU path needs explicit sync for honest timing
        torch.cuda.synchronize() # drain the stream before timing
    ts = [] # per-iter seconds
    with torch.no_grad(): # no graph for timing
        for _ in range(iters): # timed iters
            t0 = time.time() # start
            _ = m(xb) # one forward (prefill-shaped: b*CTX tokens)
            if DEV == 'cuda': # sync per iter on GPU
                torch.cuda.synchronize() # wait for kernels
            ts.append(time.time() - t0) # elapsed seconds
    med = float(np.median(ts)) # median resists one slow iter
    toks = b * CTX / med # tokens per second at this shape
    return med * 1000, toks # (ms per forward, tok/s)
FORMATS = [ # (name, bytes/param, weight transform): ordered wide -> narrow
    ('FP32', 4.0, lambda w: w), # baseline: no quantization
    ('BF16', 2.0, lambda w: fake_float_cast(w, torch.bfloat16)), # brain-float: big range, short mantissa
    ('FP16', 2.0, lambda w: fake_float_cast(w, torch.float16)), # half: tighter range than bf16 (watch overflow)
    ('FP8-E4M3', 1.0, lambda w: fake_float_cast(w, torch.float8_e4m3fn)), # 8-bit float: needs per-tensor scaling in practice; fake = round-trip
    ('INT8', 1.0, lambda w: fake_int8(w)), # symmetric per-tensor int8
    ('INT4-g32', 0.5, lambda w: fake_int4_group(w, 32)), # 4-bit, group 32 along last dim
    ('NVFP4-sim', 0.5, lambda w: fake_int4_group(w, 16)), # NVFP4 microscaling ~ int4 block-16 with fp8 scales (simulated by g16)
] # end of format ladder (memory falls 4.0 -> 0.5 B/param down the rows)
rows = [] # one row per format
base_ce = val_ce(model) # fp32 reference quality
print(f"FP32 reference val CE={base_ce:.4f} ppl={math.exp(base_ce):.2f}") # reference line
for name, bpt, fn in FORMATS: # apply each format to a clone and score it
    mq = quantize_state(model, fn) # fake-quantized clone (baseline untouched)
    ce = val_ce(mq) # val quality under this format
    ms, tps = bench_fwd(mq) # latency + throughput at (b8, ctx64)
    mem = P * bpt / 1e6 # weight memory in MB (scales/metadata ignored: noted in README)
    with torch.no_grad(): # no graph for the reconstruction-error probe
        num, den = 0.0, 0 # accumulate abs error mass + element count
        for p0, p1 in zip(model.parameters(), mq.parameters()): # paired fp32 vs quantized tensors
            num += (p0.float() - p1.float()).abs().sum().item() # abs reconstruction error
            den += p0.numel() # element count
    werr = num / max(den, 1) # mean abs weight error (shows the granularity ladder even when val CE sits in batch noise)
    rows.append({'format': name, 'bytes_param': bpt, 'mem_MB': round(mem, 3), # identity + memory columns
                 'val_CE': round(ce, 4), 'ppl': round(math.exp(ce), 2), 'dCE_vs_fp32': round(ce - base_ce, 4), # quality columns
                 'werr': round(werr, 6), # reconstruction-error column (the clean ladder signal)
                 'ms_fwd_b8': round(ms, 2), 'tok_s': round(tps, 0)}) # latency columns (CPU fake-quant: dequant overhead included)
    print(rows[-1]) # one line per format
assert rows[1]['dCE_vs_fp32'] < 0.05 # BF16 must be ~lossless (else the probe/train is broken)
assert rows[4]['mem_MB'] < rows[0]['mem_MB'] / 3.5 # INT8 must quarter memory (~4x)
assert rows[5]['mem_MB'] < rows[0]['mem_MB'] / 7.0 # INT4 must eighth memory (~8x)
with open('./results/quant.csv', 'w', newline='') as f: # save the frontier table
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())) # header from keys
    w.writeheader() # header once
    w.writerows(rows) # one row per format
print('saved ./results/quant.csv') # confirm artifact


In [ ]:
# ---- what-if: INT4 group-size sweep (per-tensor -> g128 -> g32 -> g16) + clip ablation ----
def coarse_int4(w): # per-tensor 4-bit: one scale for the whole tensor (worst case, outliers dominate)
    s = _per_tensor_scale(w, 4) # single scale from global max
    return ((w / s).round().clamp(-8, 7) * s).to(w.dtype) # quantize/dequantize coarsely
sweep = [] # sweep rows
for gname, g in [('per-tensor', 0), ('g128', 128), ('g32', 32), ('g16', 16)]: # granularity ladder: finer groups track outliers better
    fn = coarse_int4 if g == 0 else (lambda w, _g=g: fake_int4_group(w, _g)) # pick coarse vs grouped transform (default-arg captures g)
    mq = quantize_state(model, fn) # quantized clone at this granularity
    ce = val_ce(mq, n=12) # shorter val (4 clones: keep runtime low)
    with torch.no_grad(): # no graph for the error probe
        num = sum((p0.float() - p1.float()).abs().sum().item() for p0, p1 in zip(model.parameters(), mq.parameters())) # abs error mass
    sweep.append({'granularity': gname, 'clip': 'none', 'val_CE': round(ce, 4), # identity columns
                  'dCE_vs_fp32': round(ce - base_ce, 4), 'werr': round(num / P, 6)}) # quality damage + reconstruction error
    print(sweep[-1]) # one line per granularity
mq = quantize_state(model, lambda w: fake_int4_group(w, 32, clip=0.99)) # g32 + 99th-pct per-group clip (outlier control)
ce = val_ce(mq, n=12) # same short val for the clip row
with torch.no_grad(): # no graph for the error probe
    num = sum((p0.float() - p1.float()).abs().sum().item() for p0, p1 in zip(model.parameters(), mq.parameters())) # abs error mass
sweep.append({'granularity': 'g32+clip99', 'clip': '0.99', 'val_CE': round(ce, 4), # identity columns
              'dCE_vs_fp32': round(ce - base_ce, 4), 'werr': round(num / P, 6)}) # quality damage + reconstruction error
print(sweep[-1]) # clip row
with open('./results/quant_groups.csv', 'w', newline='') as f: # save the sweep
    w = csv.DictWriter(f, fieldnames=list(sweep[0].keys())) # header from keys
    w.writeheader() # header once
    w.writerows(sweep) # one row per setting
print('saved ./results/quant_groups.csv') # confirm artifact
fig, ax = plt.subplots(1, 2, figsize=(11, 4)) # two panels: frontier + granularity
mems = [r['mem_MB'] for r in rows] # x: memory per format
ces = [r['val_CE'] for r in rows] # y: quality per format
ax[0].scatter(mems, ces) # frontier scatter (down-left is better)
for r in rows: # label every point
    ax[0].annotate(r['format'], (r['mem_MB'], r['val_CE']), fontsize=8) # format name at its point
ax[0].set_xlabel('weight memory (MB)') # x label
ax[0].set_ylabel('val CE (lower better)') # y label
ax[0].set_title('P16: memory vs quality frontier') # panel title
ax[0].grid(True, alpha=0.3) # light grid
names = [s['granularity'] for s in sweep] # x labels for sweep
werrs = [s['werr'] for s in sweep] # y: mean abs weight error (clean ladder signal; val-dCE sits in batch noise at 60 iters)
ax[1].bar(names, werrs) # bar per granularity
ax[1].set_ylabel('mean abs werr (lower better)') # y label
ax[1].set_title('P16: INT4 granularity ladder (werr)') # panel title
ax[1].tick_params(axis='x', rotation=15) # tilt labels so they fit
fig.tight_layout() # avoid overlap
fig.savefig('./results/quant.png', dpi=120) # save figure
print('saved ./results/quant.png') # confirm artifact


Cleanup / next steps:
- Artifacts: `results/quant.csv` (7-format frontier), `results/quant_groups.csv` (granularity sweep),
  `results/quant.png`, `results/tinyshakespeare.txt` (cache when online).
- Latency column is CPU fake-quant (dequant runs in fp32): it measures quality + memory
  honestly, but NOT kernel speed — real INT4/FP8 wins show up on GPU/NPU kernels, rerun P3-style bench there.
- Next: serve this model behind an API with streaming + batching (P17), then schedule many
  requests the vLLM way (P18: continuous batching + PagedAttention).
